In [1]:
import numpy as np
import pandas as pd

def generate_table4_sensitivity_analysis() -> pd.DataFrame:

    a, b, t, P, C, Cw, Q = 200, 1.0, 3, 100, 40, 2, 200
    sigma = 20
    delta_val = 2.0


    np.random.seed(42)
    epsilon = np.random.normal(0, sigma, 10000)

    def evaluate_expected_z(lmbda_val: float, d_val: float, alpha_val: float) -> tuple:

        P_disc = P * (1 - d_val)
        D0_disc = a - b * P_disc - delta_val * t
        D_stoch_disc = np.maximum(D0_disc + epsilon, 0)
        Y_disc = np.minimum(Q, D_stoch_disc)
        Z_disc = np.mean(P_disc * Y_disc - C * Q - (Cw + lmbda_val) * (Q - Y_disc))


        if alpha_val <= 0.75:
            theta_bundle = 0.10
        else:
            k_steepness = 30
            theta_bundle = 0.10 * np.exp(-k_steepness * (alpha_val - 0.75))

        P_bundle = alpha_val * P
        D0_bund = a - b * P_bundle - delta_val * t
        D_stoch_bund = np.maximum((1 + theta_bundle) * D0_bund + epsilon, 0)
        Y_bund = np.minimum(Q, D_stoch_bund)
        Z_bund = np.mean(P_bundle * Y_bund - C * Q - (Cw + lmbda_val) * (Q - Y_bund))


        P_bogo = 50.0
        theta_bogo = 0.20
        D0_bogo = a - b * P_bogo - delta_val * t
        D_stoch_bogo = np.maximum((1 + theta_bogo) * D0_bogo + epsilon, 0)
        Y_bogo = np.minimum(Q, D_stoch_bogo)
        Z_bogo = np.mean(P_bogo * Y_bogo - C * Q - (Cw + lmbda_val) * (Q - Y_bogo))


        results = {'Discount': Z_disc, 'Bundle': Z_bund, 'BOGO': Z_bogo}
        optimal_strategy = max(results, key=results.get)

        return Z_disc, Z_bund, Z_bogo, optimal_strategy


    scenarios = [
        ("Waste Penalty (λ)", "λ = 2 (Low/Base)", 2.0, 0.05, 0.75),
        ("Waste Penalty (λ)", "λ = 12 (Medium)", 12.0, 0.05, 0.75),
        ("Waste Penalty (λ)", "λ = 20 (High)", 20.0, 0.05, 0.75),

        ("Discount Depth (d)", "d = 0.00 (No Markdown)", 2.0, 0.00, 0.75),
        ("Discount Depth (d)", "d = 0.05 (Local Peak / Base)", 2.0, 0.05, 0.75),
        ("Discount Depth (d)", "d = 0.15 (Medium)", 2.0, 0.15, 0.75),
        ("Discount Depth (d)", "d = 0.20 (Aggressive)", 2.0, 0.20, 0.75),

        ("Bundle Intensity (α)", "α = 0.65 (Margin Erosion)", 2.0, 0.05, 0.65),
        ("Bundle Intensity (α)", "α = 0.75 (Optimal Peak)", 2.0, 0.05, 0.75),
        ("Bundle Intensity (α)", "α = 0.85 (Framing Breakdown)", 2.0, 0.05, 0.85),
        ("Bundle Intensity (α)", "α = 0.95 (Neutralization)", 2.0, 0.05, 0.95)
    ]


    table_data = []
    for category, param_label, lmbda, d, alpha in scenarios:
        z_disc, z_bund, z_bogo, optimal = evaluate_expected_z(lmbda, d, alpha)
        table_data.append({
            "Exogenous Parameter": category,
            "Calibrated Value": param_label,
            "Discount Expected E[Z]": f"{z_disc:.2f}",
            "Bundle Expected E[Z]": f"{z_bund:.2f}",
            "BOGO Expected E[Z]": f"{z_bogo:.2f}",
            "Optimal Strategy (s*)": optimal
        })

    return pd.DataFrame(table_data)

if __name__ == "__main__":
    df_table5 = generate_table5_sensitivity_analysis()
    print(df_table5.to_string(index=False))

 Exogenous Parameter             Calibrated Value Discount Expected E[Z] Bundle Expected E[Z] BOGO Expected E[Z] Optimal Strategy (s*)
   Waste Penalty (λ)             λ = 2 (Low/Base)                 996.77              1537.58             485.08                Bundle
   Waste Penalty (λ)              λ = 12 (Medium)                 -13.66               846.13             204.54                Bundle
   Waste Penalty (λ)                λ = 20 (High)                -822.00               292.97             -19.89                Bundle
  Discount Depth (d)       d = 0.00 (No Markdown)                 971.56              1537.58             485.08                Bundle
  Discount Depth (d) d = 0.05 (Local Peak / Base)                 996.77              1537.58             485.08                Bundle
  Discount Depth (d)            d = 0.15 (Medium)                 897.20              1537.58             485.08                Bundle
  Discount Depth (d)        d = 0.20 (Aggressive)      